# RetinoGrade — train the final model on Colab (fallback for a stuck Kaggle GPU queue)

Trains **exactly** the frozen recipe that Kaggle kernel `rg-train-mix-b-full` would have run:
B = EfficientNetB0 5-class softmax, 384×384, APTOS v2 train pool (2,879) + 8,000 EyePACS "rest" images,
seed 42, fine-tune 9 epochs on the 15-epoch cosine schedule, then scores the EyePACS **dev** patients.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

Steps: (1) pin TensorFlow to the Kaggle version → (2) restart → (3) paste your Kaggle token (never saved in the notebook)
→ (4) download code + prepared data from our private Kaggle outputs → (5) train (~1–2 h) → (6) upload the model
to the private Kaggle dataset `omarashour206/rg-final-model` → (7) delete the token.

No competition images are written anywhere except this temporary Colab machine.

In [ ]:
# 1. GPU check + install the SAME versions as the Kaggle image (the saved model must load in deployment)
!nvidia-smi -L
!pip install -q "tensorflow[and-cuda]==2.21.0" "keras==3.13.2" "kaggle==2.2.5" "opencv-python-headless==4.14.0.94"
print("Installed. The session will now restart automatically - then run the next cell.")
import os; os.kill(os.getpid(), 9)

In [ ]:
# 2. Verify versions + GPU after the restart
import tensorflow as tf, keras
print("TF", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))
assert tf.__version__.startswith("2.21") and tf.config.list_physical_devices("GPU"), "need TF 2.21 + a GPU runtime"

In [ ]:
# 3. Kaggle token: typed here, kept only in this temporary machine, deleted in the last cell
import os, getpass, pathlib
token = getpass.getpass("Kaggle API token (kaggle.com -> Settings -> API): ").strip()
os.environ["KAGGLE_API_TOKEN"] = token
p = pathlib.Path.home() / ".kaggle"; p.mkdir(exist_ok=True)
(p / "access_token").write_text(token); os.chmod(p / "access_token", 0o600)
del token
!kaggle datasets list --mine | head -5

In [ ]:
# 4a. Code (private dataset omarashour206/retinograde-code)
!rm -rf /content/code && mkdir -p /content/code
!kaggle datasets download omarashour206/retinograde-code -p /content/code --unzip
import glob, zipfile, os
for z in glob.glob("/content/code/**/src.zip", recursive=True):
    zipfile.ZipFile(z).extractall("/content/code")
src = next(os.path.dirname(p) for p in glob.glob("/content/code/**/retinograde_inference.py", recursive=True))
print("code at", src, "|", open(glob.glob("/content/code/**/CODE_VERSION.txt", recursive=True)[0]).read().strip())

In [ ]:
# 4b. Prepared data from our private Kaggle kernel outputs (~10 GB, a few minutes)
!mkdir -p /content/data
!kaggle kernels output omarashour206/rg-data-v2-384 -p /content/data --page-size 200 --file-pattern "aptos_v2_.*"
!kaggle kernels output omarashour206/rg-eyepacs-data -p /content/data --page-size 200 --file-pattern "eyepacs_(rest|dev)_.*"
!ls -la /content/data
import numpy as np
for f in ["aptos_v2_u8.npy", "eyepacs_rest_u8.npy", "eyepacs_dev_u8.npy"]:
    print(f, np.load(f"/content/data/{f}", mmap_mode="r").shape)

In [ ]:
# 5. Train the final model (identical config to Kaggle rg-train-mix-b-full)
import os, sys, json
os.environ["RG_DATA_ROOTS"] = "/content/data:/content/code"     # where find_file looks (instead of /kaggle/...)
sys.path[:0] = [src, os.path.join(src, "training")]
import train
CONFIG = {"candidate": "B", "img_size": 384, "extra_data": "eyepacs", "extra_n": 8000, "eval_eyepacs_dev": True,
          "seed": 42, "folds": [-1], "ft_stop_epoch": 9, "data_dir": "/content/data", "out_dir": "/content/runs"}
rec = train.main(CONFIG)[0]
print(json.dumps({k: rec[k] for k in ("run_id", "code_version", "duration_s", "extra_data")}, indent=1))
print("EyePACS dev QWK:", rec["eyepacs_dev_metrics"]["qwk"])

In [ ]:
# 6. Upload model + small outputs to the PRIVATE Kaggle dataset omarashour206/rg-final-model
import shutil, json, os, glob, tensorflow as tf, subprocess
run = "/content/runs/B_full_s42_384_mix"
up = "/content/upload"                       # files flat at the dataset root (no sub-folders)
shutil.rmtree(up, ignore_errors=True); os.makedirs(up)
for f in os.listdir(run):
    shutil.copy(os.path.join(run, f), up)
with open(os.path.join(up, "COLAB_ENV.txt"), "w") as f:
    f.write(f"trained on Google Colab | TF {tf.__version__} | GPU {tf.config.list_physical_devices('GPU')}
")
json.dump({"title": "rg-final-model", "id": "omarashour206/rg-final-model", "licenses": [{"name": "other"}]},
          open("/content/upload/dataset-metadata.json", "w"))
print(os.listdir(up))
r = subprocess.run(["kaggle", "datasets", "create", "-p", "/content/upload", "-r", "skip"], capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
if "already" in (r.stdout + r.stderr).lower():
    r = subprocess.run(["kaggle", "datasets", "version", "-p", "/content/upload", "-r", "skip", "-m", "final model"],
                       capture_output=True, text=True)
    print(r.stdout[-800:], r.stderr[-800:])

In [ ]:
# 7. Delete the token from this machine
import os, pathlib
os.environ.pop("KAGGLE_API_TOKEN", None)
(pathlib.Path.home() / ".kaggle" / "access_token").unlink(missing_ok=True)
print("token removed - tell Claude Code: 'colab done'")